# Bangkok Validation

This notebook continues from Bangkok data preprocess and implements Sections 3–8 of `BANGKOK_ENRICHMENT_GUIDE.html`:
retrieve the HeatReady API data, create Arms 1–2, match them to AirBKK, build Arm 3 from ERA5-Land,
split train/test data by district, and report Tmax and Tmin RMSE.

Station observations remain local; no station temperature values are sent to HeatReady or Open-Meteo.
Open-Meteo results retrieved at district centroids are stored in a local cache to support repeat runs.
Run the cells in order from top to bottom.

## Step 0 — Imports and files used

The Jupyter kernel must include pandas, numpy, geopandas, requests, scikit-learn, and IPython.
If a package is missing, install it in the same kernel before continuing, for example: `%pip install geopandas scikit-learn requests`.

- Ground truth: Bangkok Validation/AirBKK_daily_tmax_tmin_guide.csv
- HeatReady snapshot for comparison: API Extraction/Heatrisk_Result/bangkok_heat_risk_20260601_20260902.local.json
- Credentials: API Extraction/heatready_credentials.local.json
- Polygon: Thailand Polygon/bangkok_districts.geojson
- Open-Meteo cache (created when the notebook runs): Bangkok Validation/era5_land_centroids_20260601_20260902.local.json

The notebook does not modify the source files or the existing JSON snapshot.

In [4]:
from pathlib import Path
import json
import random
import time

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor


def find_project_root(start):
    for path in (start, *start.parents):
        if (path / "Bangkok Validation").is_dir() and (path / "API Extraction").is_dir():
            return path
    raise FileNotFoundError("Open Jupyter from within the Crisis Ready HeatriskAPI project")


ROOT = find_project_root(Path.cwd().resolve())
VALIDATION_DIR = ROOT / "Bangkok Validation"
DAILY_PATH = VALIDATION_DIR / "AirBKK_daily_tmax_tmin_guide.csv"
BOUNDARY_PATH = ROOT / "Thailand Polygon" / "bangkok_districts.geojson"
CREDENTIALS_PATH = ROOT / "API Extraction" / "heatready_credentials.local.json"
SNAPSHOT_PATH = ROOT / "API Extraction" / "Heatrisk_Result" / "bangkok_heat_risk_20260601_20260902.local.json"
GRID_CACHE_PATH = VALIDATION_DIR / "era5_land_centroids_20260601_20260902.local.json"

HEATREADY_URL = "https://5trhrgas69.execute-api.us-east-1.amazonaws.com/v1/evaluate"
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
PROJECT_ID = "bangkok-districts-50"
DATE_FROM = "2026-06-01"
DATE_TO = "2026-09-02"
PAGE_LIMIT = 500
EXPECTED_DATES = pd.date_range(DATE_FROM, DATE_TO).strftime("%Y-%m-%d").tolist()

print("Project:", PROJECT_ID)
print("Validation period:", DATE_FROM, "to", DATE_TO)
print("Expected dates:", len(EXPECTED_DATES))

Project: bangkok-districts-50
Validation period: 2026-06-01 to 2026-09-02
Expected dates: 94


## Step 1 — Retrieve HeatReady data according to Guide Section 3

Read the username and key from a local file without displaying the key.
Call the `evaluate-heat-risk` action through POST with `project_id`, `date_from`, and `date_to`,
using a limit of 2000 and incrementing the offset until `total_rows` is reached.
Set `include_forecast=False` because this validation uses historical data.
Stop immediately if the API returns an empty page before completion or if the row count does not match.

In [5]:
# Load credentials into memory only; do not display the key in the output
if not CREDENTIALS_PATH.exists():
    raise FileNotFoundError(CREDENTIALS_PATH)

credentials = json.loads(CREDENTIALS_PATH.read_text(encoding="utf-8"))
if not credentials.get("username") or not credentials.get("key"):
    raise ValueError("Credentials must contain username and key")


def call(action, payload):
    body = {
        "username": credentials["username"],
        "key": credentials["key"],
        "action": action,
        "payload": payload,
    }
    for attempt in range(4):
        response = requests.post(HEATREADY_URL, json=body, timeout=120)
        try:
            result = response.json()
        except ValueError:
            raise RuntimeError("HeatReady returned a non-JSON response")

        if response.status_code in {429, 500, 502, 503, 504} and attempt < 3:
            time.sleep(5 * (2 ** attempt))
            continue
        if not response.ok or result.get("error"):
            raise RuntimeError(
                f"HeatReady request failed: HTTP {response.status_code}, "
                f"code={result.get('code')}, error={result.get('error')}"
            )
        return result
    raise RuntimeError("HeatReady request failed after retries")


# Pagination from Section 3: retrieve pages until all total_rows have been received
def fetch_metrics(project_id, date_from, date_to):
    rows, offset, total_rows = [], 0, None
    while total_rows is None or offset < total_rows:
        result = call("evaluate-heat-risk", {
            "project_id": project_id,
            "date_from": date_from,
            "date_to": date_to,
            "include_forecast": False,
            "limit": PAGE_LIMIT,
            "offset": offset,
        })
        page = result.get("metrics", [])
        total_rows = int(result["total_rows"])
        if not page and offset < total_rows:
            raise RuntimeError("HeatReady returned an empty page before all rows were retrieved")
        rows.extend(page)
        offset += len(page)
        print(f"HeatReady: {len(rows):,}/{total_rows:,} rows")
    if len(rows) != total_rows:
        raise RuntimeError("The number of metrics does not match total_rows")
    return rows


api_rows = fetch_metrics(PROJECT_ID, DATE_FROM, DATE_TO)
print("Fetched metrics:", len(api_rows))

HeatReady: 500/4,700 rows
HeatReady: 1,000/4,700 rows
HeatReady: 1,500/4,700 rows
HeatReady: 2,000/4,700 rows
HeatReady: 2,500/4,700 rows
HeatReady: 3,000/4,700 rows
HeatReady: 3,500/4,700 rows
HeatReady: 4,000/4,700 rows
HeatReady: 4,500/4,700 rows
HeatReady: 4,700/4,700 rows
Fetched metrics: 4700


## Step 2 — Validate the API response and compare it with the JSON snapshot

Check district-date keys, the date range, and `data_source` from the actual rows.
The Guide uses only the ERA5 band for the primary evaluation, even when `include_forecast=False`,
so `data_source` must be checked again after retrieval.

Use the existing JSON snapshot to identify whether the current retrieval differs from the saved snapshot.
If values differ, the notebook reports the number of changed rows but continues with the current API response.

In [6]:
# Inspect the retrieved rows because include_forecast=False does not guarantee that every row is ERA5
required_api_fields = {"name", "date", "data_source", "day_t2m_max", "day_t2m_min"}
if any(not required_api_fields.issubset(row) for row in api_rows):
    raise ValueError("Some API metric rows are missing required fields")
api_keys = [(row["name"], row["date"]) for row in api_rows]
if len(api_keys) != len(set(api_keys)):
    raise ValueError("API metrics contain duplicate name + date pairs")
if any(not DATE_FROM <= row["date"] <= DATE_TO for row in api_rows):
    raise ValueError("API metrics contain dates outside the requested range")

print("Rows by data_source:")
display(pd.Series([row["data_source"] for row in api_rows]).value_counts().rename("rows").to_frame())
print("API names:", len({row["name"] for row in api_rows}))
print("API dates:", len({row["date"] for row in api_rows}))

if SNAPSHOT_PATH.exists():
    snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding="utf-8"))
    old_rows = snapshot["metrics"]
    old_by_key = {(row["name"], row["date"]): row for row in old_rows}
    new_by_key = {(row["name"], row["date"]): row for row in api_rows}
    changed = sum(
        row["day_t2m_max"] != old_by_key[key]["day_t2m_max"]
        or row["day_t2m_min"] != old_by_key[key]["day_t2m_min"]
        or row["data_source"] != old_by_key[key]["data_source"]
        or row.get("downscaled") != old_by_key[key].get("downscaled")
        for key, row in new_by_key.items() if key in old_by_key
    )
    print("Snapshot keys only:", len(old_by_key.keys() - new_by_key.keys()))
    print("New API keys only:", len(new_by_key.keys() - old_by_key.keys()))
    print("Shared keys with changed raw/source/downscaled data:", changed)

Rows by data_source:


,rows
era5,4700


API names: 50
API dates: 94
Snapshot keys only: 0
New API keys only: 0
Shared keys with changed raw/source/downscaled data: 4694


## Step 3 — Create Arm 1 and Arm 2 according to Section 4

Arm 1 is raw ERA5-Land from `day_t2m_max` and `day_t2m_min`.
Arm 2 is the corrected global-model value from `downscaled.metrics`.

Values in `downscaled.metrics` with another `base_source`, or with a null `delta_c`,
are not qualifying corrections and are therefore not included in Arm 2.
Check Tmax and Tmin separately because their corrections may not qualify at the same time.

In [7]:
# Arm 2 requires a qualifying correction for the specific target; metrics with delta_c=None cannot be substituted
def split_arms(rows):
    era5, global_model = [], []
    for row in rows:
        if row["data_source"] != "era5":
            continue
        era5.append({
            "name": row["name"], "date": row["date"],
            "tmax": row["day_t2m_max"], "tmin": row["day_t2m_min"],
        })
        downscaled = row.get("downscaled")
        if downscaled and downscaled.get("base_source") == "era5_land":
            entry = {"name": row["name"], "date": row["date"]}
            if downscaled["tmax"]["delta_c"] is not None:
                entry["tmax"] = downscaled["metrics"]["day_t2m_max"]
            if downscaled["tmin"]["delta_c"] is not None:
                entry["tmin"] = downscaled["metrics"]["day_t2m_min"]
            if "tmax" in entry or "tmin" in entry:
                global_model.append(entry)
    return era5, global_model


era5_rows, global_rows = split_arms(api_rows)
raw_df = pd.DataFrame(era5_rows).rename(
    columns={"tmax": "raw_tmax", "tmin": "raw_tmin"}
)
global_df = pd.DataFrame(global_rows).rename(
    columns={"tmax": "global_tmax", "tmin": "global_tmin"}
)
if raw_df.empty or global_df.empty:
    raise ValueError("Arm 1 or Arm 2 has no data for the requested period")
if raw_df.duplicated(["name", "date"]).any() or global_df.duplicated(["name", "date"]).any():
    raise ValueError("Arm 1 or Arm 2 contains duplicate name + date pairs")

print("Arm 1 rows:", len(raw_df))
print("Arm 2 rows with at least one corrected target:", len(global_df))
print("Arm 2 usable tmax:", global_df["global_tmax"].notna().sum())
print("Arm 2 usable tmin:", global_df["global_tmin"].notna().sum())

Arm 1 rows: 4700
Arm 2 rows with at least one corrected target: 4032
Arm 2 usable tmax: 4032
Arm 2 usable tmin: 4032


## Step 4 — Load ground truth from preprocessing

Use only `name`, `date`, `tmax`, and `tmin` values that passed the physical-plausibility check and
the completeness threshold of at least 20 readings per station-day.
Do not fill missing district-days with zero or remove an entire district solely because its full-period coverage is low.

In [8]:
# This file has already passed hourly QC and the threshold of at least 20 readings per station-day
daily = pd.read_csv(DAILY_PATH, encoding="utf-8-sig")
if list(daily.columns) != ["name", "date", "tmax", "tmin"]:
    raise ValueError("Ground truth must contain the name,date,tmax,tmin columns produced by preprocessing")
daily["date"] = pd.to_datetime(daily["date"], errors="raise").dt.strftime("%Y-%m-%d")
if daily.isna().any().any() or daily.duplicated(["name", "date"]).any():
    raise ValueError("Ground truth contains null values or duplicate name + date pairs")
if not (daily["tmin"] <= daily["tmax"]).all():
    raise ValueError("Ground truth contains tmin greater than tmax")
if any(not DATE_FROM <= day <= DATE_TO for day in daily["date"]):
    raise ValueError("Ground truth contains dates outside the validation period")

print("Ground truth rows:", len(daily))
print("Districts with observations:", daily["name"].nunique())
print("Dates with observations:", daily["date"].nunique())
display(daily.head())

Ground truth rows: 4521
Districts with observations: 49
Dates with observations: 94


,name,date,tmax,tmin
0,Bang Bon,2026-06-01,35.1,25.7
1,Bang Bon,2026-06-02,36.6,27.6
2,Bang Bon,2026-06-03,34.7,27.8
3,Bang Bon,2026-06-04,34.0,29.3
4,Bang Bon,2026-06-05,34.3,29.3


## Step 5 — Match HeatReady data to station observations

Use `name + date` as the shared key for both datasets.
Check how many Arm 1 rows match ground truth and how many qualifying Arm 2 corrections are available.
Rows without a station observation are not included in training or RMSE.

In [9]:
# Start from ground truth so every station row that fails to match the API remains visible
station_api = daily.merge(
    raw_df, on=["name", "date"], how="left", validate="one_to_one", indicator=True
)
unmatched_raw = station_api.loc[station_api["_merge"] != "both", ["name", "date"]]
print("Ground truth matched to raw ERA5:", len(station_api) - len(unmatched_raw))
print("Ground truth without raw ERA5:", len(unmatched_raw))
if len(unmatched_raw):
    display(unmatched_raw.head(20))
    raise ValueError("Ground truth does not fully match Arm 1")

station_global = daily[["name", "date"]].merge(
    global_df, on=["name", "date"], how="left", validate="one_to_one"
)
print("Ground truth with global tmax correction:", station_global["global_tmax"].notna().sum())
print("Ground truth with global tmin correction:", station_global["global_tmin"].notna().sum())

Ground truth matched to raw ERA5: 4521
Ground truth without raw ERA5: 0
Ground truth with global tmax correction: 3876
Ground truth with global tmin correction: 3876


## Step 6 — Calculate district centroids for Arm 3

The Guide retrieves ERA5-Land at 0.1° resolution at each polygon centroid.
Use the same boundaries as the project and derive `name` from `amp_en` in the same way as the project-creation notebook.
Calculate centroids after projecting to UTM zone 47N, then convert the coordinates back to WGS84.
Retrieve only the 49 districts with ground truth to avoid an unnecessary request for Bang Kapi.

In [10]:
boundaries = gpd.read_file(BOUNDARY_PATH)
if len(boundaries) != 50 or not {"amp_en", "geometry"}.issubset(boundaries.columns):
    raise ValueError("Boundary data must contain 50 districts and the amp_en and geometry columns")
if boundaries.crs is None:
    raise ValueError("Boundary data has no CRS")
boundaries["name"] = boundaries["amp_en"].fillna("").astype(str).str.strip()
if boundaries["name"].eq("").any() or boundaries["name"].duplicated().any():
    raise ValueError("Boundary data contains blank or duplicate district names")

# Calculate centroids in a projected CRS before converting them back to API latitude/longitude
projected = boundaries.to_crs("EPSG:32647")
centroid_geometry = projected.geometry.centroid.to_crs("EPSG:4326")
centroids = pd.DataFrame({
    "name": boundaries["name"].to_numpy(),
    "latitude": centroid_geometry.y.to_numpy().round(6),
    "longitude": centroid_geometry.x.to_numpy().round(6),
})
centroids = centroids.loc[centroids["name"].isin(daily["name"].unique())].sort_values("name")
if set(centroids["name"]) != set(daily["name"]):
    raise ValueError("District names in ground truth do not match the boundary data")
print("Centroids to request:", len(centroids))
display(centroids.head())

Centroids to request: 49


,name,latitude,longitude
10,Bang Bon,13.656392,100.391689
2,Bang Khae,13.715323,100.395306
35,Bang Khen,13.866331,100.630124
22,Bang Kho Laem,13.696641,100.510604
12,Bang Khun Thian,13.589031,100.428046


## Step 7 — Retrieve ERA5-Land from Open-Meteo and cache the results

The Guide refers to `src/open_meteo.py`, which is not available in this workspace.
This notebook therefore calls the Historical Weather API directly with `models=era5_land` (0.1°),
daily temperature_2m_max/min, timezone=Asia/Bangkok, cell_selection=nearest
and `elevation=nan` so the service does not adjust temperature for elevation.

One request per centroid covers all 94 days.
The cache contains only district coordinates and grid values; it contains no station temperature observations.
If the query or centroids differ from the existing cache, the notebook stops to avoid using the wrong dataset.

In [11]:
GRID_PARAMS = {
    "start_date": DATE_FROM,
    "end_date": DATE_TO,
    "daily": "temperature_2m_max,temperature_2m_min",
    "models": "era5_land",
    "timezone": "Asia/Bangkok",
    "temperature_unit": "celsius",
    "cell_selection": "nearest",
    "elevation": "nan",
}
# Bind the cache to the exact parameters and centroids so repeat runs cannot use the wrong grid dataset
grid_query = {
    "params": GRID_PARAMS,
    "centroids": centroids.to_dict("records"),
}


def fetch_centroid_grid(name, latitude, longitude):
    params = {**GRID_PARAMS, "latitude": latitude, "longitude": longitude}
    for attempt in range(3):
        response = requests.get(OPEN_METEO_URL, params=params, timeout=120)
        if response.status_code in {429, 500, 502, 503, 504} and attempt < 2:
            time.sleep(2 ** attempt)
            continue
        response.raise_for_status()
        payload = response.json()
        if payload.get("error"):
            raise RuntimeError(f"Open-Meteo error for {name}: {payload.get('reason')}")
        if payload.get("timezone") != "Asia/Bangkok":
            raise ValueError(f"Open-Meteo timezone does not match for {name}")
        data = payload.get("daily", {})
        days = data.get("time", [])
        tmax = data.get("temperature_2m_max", [])
        tmin = data.get("temperature_2m_min", [])
        if days != EXPECTED_DATES or len(tmax) != len(days) or len(tmin) != len(days):
            raise ValueError(f"Open-Meteo returned incomplete dates or value counts for {name}")
        if any(a is None or b is None for a, b in zip(tmax, tmin)):
            raise ValueError(f"Open-Meteo returned null values for {name}")
        return [
            {"name": name, "date": day, "grid_tmax": a, "grid_tmin": b}
            for day, a, b in zip(days, tmax, tmin)
        ]
    raise RuntimeError(f"Open-Meteo request failed for {name}")


if GRID_CACHE_PATH.exists():
    cached = json.loads(GRID_CACHE_PATH.read_text(encoding="utf-8"))
    if cached.get("query") != grid_query:
        raise ValueError("Grid cache uses a different query or set of centroids; inspect the cache before running")
    grid_rows = cached["rows"]
    print("Loaded grid cache:", GRID_CACHE_PATH)
else:
    grid_rows = []
    for point in centroids.itertuples(index=False):
        grid_rows.extend(fetch_centroid_grid(point.name, point.latitude, point.longitude))
        print(f"Grid fetched: {len(grid_rows):,}/{len(centroids) * len(EXPECTED_DATES):,}")
    GRID_CACHE_PATH.write_text(
        json.dumps({"query": grid_query, "rows": grid_rows}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved grid cache:", GRID_CACHE_PATH)

grid = pd.DataFrame(grid_rows)
if set(grid.columns) != {"name", "date", "grid_tmax", "grid_tmin"}:
    raise ValueError("Grid cache columns do not match the expected schema")
if grid.duplicated(["name", "date"]).any():
    raise ValueError("Grid contains duplicate name + date pairs")
expected_grid_keys = {(name, day) for name in centroids["name"] for day in EXPECTED_DATES}
if set(zip(grid["name"], grid["date"])) != expected_grid_keys:
    raise ValueError("Grid has missing or extra district-date rows")
if grid[["grid_tmax", "grid_tmin"]].isna().any().any():
    raise ValueError("Grid contains null temperature values")
print("Validated grid rows:", len(grid))

Grid fetched: 94/4,606
Grid fetched: 188/4,606
Grid fetched: 282/4,606
Grid fetched: 376/4,606
Grid fetched: 470/4,606
Grid fetched: 564/4,606
Grid fetched: 658/4,606
Grid fetched: 752/4,606
Grid fetched: 846/4,606
Grid fetched: 940/4,606
Grid fetched: 1,034/4,606
Grid fetched: 1,128/4,606
Grid fetched: 1,222/4,606
Grid fetched: 1,316/4,606
Grid fetched: 1,410/4,606
Grid fetched: 1,504/4,606
Grid fetched: 1,598/4,606
Grid fetched: 1,692/4,606
Grid fetched: 1,786/4,606
Grid fetched: 1,880/4,606
Grid fetched: 1,974/4,606
Grid fetched: 2,068/4,606
Grid fetched: 2,162/4,606
Grid fetched: 2,256/4,606
Grid fetched: 2,350/4,606
Grid fetched: 2,444/4,606
Grid fetched: 2,538/4,606
Grid fetched: 2,632/4,606
Grid fetched: 2,726/4,606
Grid fetched: 2,820/4,606
Grid fetched: 2,914/4,606
Grid fetched: 3,008/4,606
Grid fetched: 3,102/4,606
Grid fetched: 3,196/4,606
Grid fetched: 3,290/4,606
Grid fetched: 3,384/4,606
Grid fetched: 3,478/4,606
Grid fetched: 3,572/4,606
Grid fetched: 3,666/4,606
Grid fe

## Step 8 — Split train/test data by district (Section 6)

Shuffle district names with seed 42 as specified by the Guide.
Each district must appear only in train or test; rows from the same district are never split across both sets.
Check row counts and district names after the split without using test targets during training.

In [12]:
# Split at the district level, not the row level, so no district appears in both train and test
districts = sorted(daily["name"].unique())
random.seed(42)
random.shuffle(districts)
n_test = max(1, len(districts) // 5)
test_districts = set(districts[:n_test])
train_districts = set(districts[n_test:])
if test_districts & train_districts:
    raise AssertionError("Train and test districts overlap")

model_data = daily.merge(grid, on=["name", "date"], how="left", validate="one_to_one")
if model_data[["grid_tmax", "grid_tmin"]].isna().any().any():
    raise ValueError("Some ground-truth rows have no ERA5-Land features")

train = model_data.loc[model_data["name"].isin(train_districts)].copy()
test = model_data.loc[model_data["name"].isin(test_districts)].copy()
print(f"Train: {len(train):,} rows, {len(train_districts)} districts")
print(f"Test: {len(test):,} rows, {len(test_districts)} districts")
print("Test districts:", sorted(test_districts))
if train.empty or test.empty:
    raise ValueError("Train or test data is empty")

Train: 3,686 rows, 40 districts
Test: 835 rows, 9 districts
Test districts: ['Bang Khun Thian', 'Bangkok Noi', 'Bueng Kum', 'Khlong Sam Wa', 'Lat Krabang', 'Min Buri', 'Nong Chok', 'Phasi Charoen', 'Thon Buri']


## Step 9 — Train Arm 3: local Random Forest

Use ERA5-Land grid Tmax and Tmin as the two features, without vulnerability covariates.
Train separate Tmax and Tmin target models as specified by the Guide, using 200 trees and `random_state=42`.
Generate predictions only for test data that was never used to train the models.

In [13]:
# Use only ERA5-Land grid values as features; station Tmax/Tmin values are the targets
FEATURES = ["grid_tmax", "grid_tmin"]
model_tmax = RandomForestRegressor(n_estimators=200, random_state=42)
model_tmin = RandomForestRegressor(n_estimators=200, random_state=42)

model_tmax.fit(train[FEATURES], train["tmax"])
model_tmin.fit(train[FEATURES], train["tmin"])

local_predictions = test[["name", "date"]].copy()
local_predictions["local_tmax"] = model_tmax.predict(test[FEATURES])
local_predictions["local_tmin"] = model_tmin.predict(test[FEATURES])
print("Local predictions on held-out test rows:", len(local_predictions))
display(local_predictions.head())

Local predictions on held-out test rows: 835


,name,date,local_tmax,local_tmin
376,Bang Khun Thian,2026-06-01,34.637967,25.470552
377,Bang Khun Thian,2026-06-02,34.031443,26.194445
378,Bang Khun Thian,2026-06-03,34.454774,28.082611
379,Bang Khun Thian,2026-06-04,33.604863,28.602621
380,Bang Khun Thian,2026-06-05,34.728893,29.378257


## Step 10 — Build a test table containing all three Arms

Each row represents one station observation for one district-day.
The raw/global/local columns are predictions matched to the same observation.
Arm 2 may be missing when a correction is gated out; no replacement value is imputed.

In [ ]:
# Left joins retain every test observation and leave Arm 2 missing where no correction is available
evaluation = (
    test[["name", "date", "tmax", "tmin"]]
    .merge(raw_df, on=["name", "date"], how="left", validate="one_to_one")
    .merge(global_df, on=["name", "date"], how="left", validate="one_to_one")
    .merge(local_predictions, on=["name", "date"], how="left", validate="one_to_one")
)
if evaluation[["tmax", "tmin", "raw_tmax", "raw_tmin", "local_tmax", "local_tmin"]].isna().any().any():
    raise ValueError("Test data is missing ground truth, raw ERA5, or local predictions")
print("Test rows:", len(evaluation))
print("Global tmax available:", evaluation["global_tmax"].notna().sum())
print("Global tmin available:", evaluation["global_tmin"].notna().sum())
display(evaluation.head())

## Step 11 — Calculate two forms of RMSE separately for Tmax and Tmin

Guide-style: each Arm uses the test rows for which that Arm has a value.
Paired comparison: all three Arms use the same district-day set for the relevant target.
Always report n with RMSE because Arm 2 may have no correction for some districts.
Guide-style scores for different Arms may use different numbers of districts and days.

In [ ]:
def rmse(pred, actual):
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(actual)) ** 2)))


# Calculate guide_available and paired_common separately by target and report the pair count n
score_rows = []
for target in ("tmax", "tmin"):
    pred_cols = {
        "raw_era5": f"raw_{target}",
        "global_model": f"global_{target}",
        "local_model": f"local_{target}",
    }
    common = evaluation.dropna(subset=[target, *pred_cols.values()])
    for mode, frame in [("guide_available", evaluation), ("paired_common", common)]:
        for arm, pred_col in pred_cols.items():
            matched = frame.dropna(subset=[target, pred_col])
            if matched.empty:
                raise ValueError(f"No matched data pairs for {mode}, {target}, {arm}")
            score_rows.append({
                "mode": mode,
                "target": target,
                "arm": arm,
                "n": len(matched),
                "rmse_c": rmse(matched[pred_col], matched[target]),
            })

scores = pd.DataFrame(score_rows)
display(scores.round({"rmse_c": 3}))

## Step 12 — Report coverage and score limitations

Report the spatial and temporal coverage of ground truth before interpreting RMSE.
Bang Kapi has no observations; Suan Luang has observations only in June.
Some districts in the Guide test set have no global correction, so n and
the list of evaluated districts must be read together with each score. Both analysis forms are retained so the team can select the one appropriate to its objective.

In [ ]:
# Start coverage from all 50 boundary districts so Bang Kapi appears with zero days
coverage = boundaries[["name"]].merge(
    daily.groupby("name")["date"].nunique().rename("accepted_days"),
    on="name", how="left", validate="one_to_one",
)
coverage["accepted_days"] = coverage["accepted_days"].fillna(0).astype(int)
coverage["coverage_pct"] = (coverage["accepted_days"] / len(EXPECTED_DATES) * 100).round(1)

monthly = daily.assign(month=daily["date"].str.slice(0, 7)).groupby("month").agg(
    rows=("name", "size"),
    districts=("name", "nunique"),
    days=("date", "nunique"),
).reset_index()

test_coverage = evaluation.groupby("name").agg(
    test_days=("date", "size"),
    global_tmax_days=("global_tmax", "count"),
    global_tmin_days=("global_tmin", "count"),
).reset_index().sort_values("global_tmax_days")

print("Ground truth coverage:", len(daily), "of", 50 * len(EXPECTED_DATES), "district-days")
print("Districts with zero accepted days:", coverage.loc[coverage["accepted_days"].eq(0), "name"].tolist())
print("Coverage by district:")
display(coverage.sort_values(["accepted_days", "name"]).reset_index(drop=True))
print("Coverage by month:")
display(monthly)
print("Global correction availability in held-out test districts:")
display(test_coverage.reset_index(drop=True))
print("RMSE by mode, target and Arm:")
display(scores.round({"rmse_c": 3}))

print(
    "Interpretation: paired_common compares all three Arms on identical district-days; "
    "guide_available follows the Guide's available rows per Arm. "
    "The n column shows how many comparisons support each RMSE."
)

## Step 13 — Save result tables as CSV files for reporting

Save district-day test results, RMSE by Arm and target, district coverage, and monthly coverage
to the Bangkok Validation folder in accordance with the Guide's Results and Reporting sections.
The test file contains only held-out districts; global values without a qualifying correction remain blank.
Rerunning this cell overwrites the result CSV files for the same date range.

In [ ]:
# Use the already calculated and validated tables; do not change criteria or recalculate scores
period_tag = f"{DATE_FROM.replace('-', '')}_{DATE_TO.replace('-', '')}"
exports = {
    f"bangkok_validation_test_rows_{period_tag}.csv": evaluation.sort_values(["name", "date"]),
    f"bangkok_validation_rmse_{period_tag}.csv": scores,
    f"bangkok_validation_district_coverage_{period_tag}.csv": coverage.sort_values("name"),
    f"bangkok_validation_monthly_coverage_{period_tag}.csv": monthly.sort_values("month"),
}
for filename, table in exports.items():
    output_path = VALIDATION_DIR / filename
    table.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(table):,} rows: {output_path}")

## Interpreting the results and their scope

- Report Tmax and Tmin RMSE separately rather than combining them into a single score.
- Arm 2 uses only values for which global downscaling was applied according to `delta_c`; values from another grid or station source in `downscaled.metrics` are not substituted.
- `guide_available` follows the Section 7 example, but each Arm may evaluate different district-days.
- `paired_common` compares all three Arms on the same observations; always inspect n.
- The model evaluates transfer to held-out districts using a district-level train/test split, while train and test cover the same dates.
- The centroid grid point represents the polygon, while the station is the actual measurement location within the district; this creates a spatial difference.
- This run evaluates only historical rows with `data_source=era5`; it does not evaluate recent `open_meteo` data or forecasts.
- If HeatReady/ERA5 results are published, include Copernicus/C3S attribution as specified by the Guide.